# 📝 Few-Shot Prompting

**Day 2 — Notebook 2 of 5 | Estimated Time: 25 minutes**

---

## What You'll Learn
- What few-shot prompting is and when it outperforms zero-shot
- How to structure examples for consistent outputs
- How to enforce output formats (JSON, markdown tables) using examples

---

## What is Few-Shot Prompting?

In **few-shot prompting**, you provide the model with **a few examples** of the desired input-output behavior before giving it the actual task. The model learns the pattern from your examples and applies it.

```
Zero-shot:  "Classify as positive/negative: 'I love this product'"
Few-shot:   "Examples:
             'Great quality!' → Positive
             'Terrible service' → Negative
             'It works fine' → Neutral
             
             Now classify: 'I love this product'"
```

### When to use few-shot vs zero-shot:
- **Zero-shot:** Simple, well-understood tasks
- **Few-shot:** When you need specific output formats, domain-specific patterns, or the model struggles with zero-shot

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=get_api_key())
MODEL_ID = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. Basic Few-Shot: Sentiment Classification

Let's compare zero-shot vs few-shot on the same task.

In [ ]:
# Zero-shot — sometimes gives verbose output
zero_shot_prompt = """
Classify the sentiment: "The delivery was late but the product quality was excellent."
"""

response = client.models.generate_content(model=MODEL_ID, contents=zero_shot_prompt)
print("Zero-shot result:")
print(response.text)

In [ ]:
# Few-shot — examples teach the model the desired format
few_shot_prompt = """
Classify the sentiment of each review. Respond with only one word: Positive, Negative, or Mixed.

Review: "I love this product, it works perfectly!"
Sentiment: Positive

Review: "Worst purchase ever. Complete waste of money."
Sentiment: Negative

Review: "It works, but the instructions were confusing."
Sentiment: Mixed

Review: "The delivery was late but the product quality was excellent."
Sentiment:
"""

response = client.models.generate_content(model=MODEL_ID, contents=few_shot_prompt)
print("Few-shot result:")
print(response.text.strip())

> 💡 **Notice:** Few-shot prompting gives a much more **consistent and concise** output because the model learns the pattern from the examples.

---

## 2. Enforcing JSON Output with Few-Shot

One of the most powerful uses of few-shot prompting is enforcing structured output formats.

In [ ]:
prompt = """
Extract product information and return as JSON.

Input: "The iPhone 15 Pro by Apple costs $999 and has 256GB storage."
Output: {"product": "iPhone 15 Pro", "brand": "Apple", "price": 999, "storage": "256GB"}

Input: "Samsung Galaxy S24 Ultra is priced at $1,299 with 512GB."
Output: {"product": "Galaxy S24 Ultra", "brand": "Samsung", "price": 1299, "storage": "512GB"}

Input: "Google Pixel 9 Pro costs $799 and comes with 128GB storage."
Output:
"""

response = client.models.generate_content(model=MODEL_ID, contents=prompt)
print(response.text.strip())

In [ ]:
# Verify it's valid JSON
import json

try:
    parsed = json.loads(response.text.strip())
    print("✅ Valid JSON!")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"❌ Invalid JSON: {e}")
    print("Raw output:", response.text.strip())

---

## 3. Few-Shot for Custom Classification

Few-shot is particularly useful when you need domain-specific categories.

In [ ]:
# Custom intent classification for an IT helpdesk
prompt = """
Classify the IT support ticket into one of these categories: 
Hardware, Software, Network, Access, Other

Ticket: "My laptop screen is flickering and has dead pixels."
Category: Hardware

Ticket: "I can't connect to the VPN from home."
Category: Network

Ticket: "Excel keeps crashing when I open large files."
Category: Software

Ticket: "I need access to the marketing SharePoint site."
Category: Access

Ticket: "My keyboard is not working after the coffee spill."
Category:
"""

response = client.models.generate_content(model=MODEL_ID, contents=prompt)
print(f"Category: {response.text.strip()}")

---

## 4. Few-Shot for Format Transformation

Teach the model to transform data from one format to another.

In [ ]:
# Natural language to SQL
prompt = """
Convert the natural language query to SQL. The database has tables:
- employees (id, name, department, salary, hire_date)
- departments (id, name, budget)

Query: "Show all employees in the Engineering department"
SQL: SELECT * FROM employees WHERE department = 'Engineering';

Query: "What is the average salary by department?"
SQL: SELECT department, AVG(salary) as avg_salary FROM employees GROUP BY department;

Query: "Find employees hired after January 2024 with salary above 80000"
SQL: SELECT * FROM employees WHERE hire_date > '2024-01-01' AND salary > 80000;

Query: "List the top 5 highest paid employees with their department names"
SQL:
"""

response = client.models.generate_content(model=MODEL_ID, contents=prompt)
print(response.text.strip())

---

## 5. Best Practices for Few-Shot Examples

| Practice | Why |
|----------|-----|
| Use **3-5 examples** | Enough to establish pattern, not too many to waste tokens |
| Make examples **diverse** | Cover edge cases and variations |
| Keep examples **consistent** | Same format in every example |
| Order examples **logically** | Simple → complex, or randomized |
| Be **exact** in format | If you want JSON with specific keys, show those exact keys |

---

## 🏋️ Exercise 1: Build a Product Review Analyzer

Create a few-shot prompt that analyzes product reviews and returns JSON with: sentiment, key_pros (list), key_cons (list), and rating (1-5).

In [ ]:
# Exercise 1: Product review analyzer
# TODO: Write a few-shot prompt with 2-3 examples, then test it

prompt = """
YOUR FEW-SHOT PROMPT HERE

Review: "The headphones have amazing sound quality and are very comfortable, 
but the battery life is disappointing and they're overpriced for what you get."
Analysis:
"""

response = client.models.generate_content(model=MODEL_ID, contents=prompt)
print(response.text)

---

## 🏋️ Exercise 2: Email Subject Line Generator

Create a few-shot prompt that generates professional email subject lines from email body summaries.

In [ ]:
# Exercise 2: Email subject line generator
# TODO: Write a few-shot prompt with examples of body → subject line

test_bodies = [
    "Requesting approval for the Q3 marketing budget of $50,000 for digital advertising campaigns.",
    "Informing the team that the office will be closed on Friday for building maintenance.",
    "Following up on the client meeting from yesterday and sharing the action items.",
]

for body in test_bodies:
    prompt = f"""YOUR FEW-SHOT PROMPT HERE

    Body: "{body}"
    Subject:
    """
    
    response = client.models.generate_content(model=MODEL_ID, contents=prompt)
    print(f"Body: {body[:60]}...")
    print(f"Subject: {response.text.strip()}")
    print()

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| Few-Shot Prompting | Blog | [promptingguide.ai/techniques/fewshot](https://www.promptingguide.ai/techniques/fewshot) |
| Language Models are Few-Shot Learners (GPT-3 Paper) | Paper | [arxiv.org/abs/2005.14165](https://arxiv.org/abs/2005.14165) |
| Google Prompting Strategies | Docs | [ai.google.dev/gemini-api/docs/prompting-strategies](https://ai.google.dev/gemini-api/docs/prompting-strategies) |

---

**Next up:** [03_Chain_of_Thought_Prompting.ipynb](./03_Chain_of_Thought_Prompting.ipynb) — Make the model show its reasoning